# `DATA EXTRACTION`

Quick Inspection

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving sales_data.csv to sales_data (2).csv


In [149]:
import pandas as pd
import sqlite3
import os

In [151]:
df = pd.read_csv("sales_data.csv")

In [152]:
print(df.shape)

(1000, 7)


In [153]:
print(df.head)

<bound method NDFrame.head of      order_id   customer_name              product  quantity   price  \
0        1106     Frank Brown            USB-C Hub       3.0   42.62   
1        1818      Wendy King            USB-C Hub       5.0   49.66   
2        1840  Quinn Robinson  Mechanical Keyboard       8.0   95.23   
3        1022     Samuel Hall            USB-C Hub       7.0   48.16   
4        1206     Samuel Hall  Mechanical Keyboard       5.0  101.29   
..        ...             ...                  ...       ...     ...   
995      1259   Xander Wright        Monitor Stand       7.0   37.06   
996      1379       Bob Smith           Smart Plug       8.0   16.63   
997      1703   Alice Johnson      Cable Organiser       4.0   11.46   
998      1614   Isabella Chen  Mechanical Keyboard       5.0   80.38   
999      1911     Samuel Hall           Smart Plug       2.0   17.73   

            order_date   region  
0           01/06/2023  Central  
1           09/05/2024    North  
2  

In [154]:
print(df.info)

<bound method DataFrame.info of      order_id   customer_name              product  quantity   price  \
0        1106     Frank Brown            USB-C Hub       3.0   42.62   
1        1818      Wendy King            USB-C Hub       5.0   49.66   
2        1840  Quinn Robinson  Mechanical Keyboard       8.0   95.23   
3        1022     Samuel Hall            USB-C Hub       7.0   48.16   
4        1206     Samuel Hall  Mechanical Keyboard       5.0  101.29   
..        ...             ...                  ...       ...     ...   
995      1259   Xander Wright        Monitor Stand       7.0   37.06   
996      1379       Bob Smith           Smart Plug       8.0   16.63   
997      1703   Alice Johnson      Cable Organiser       4.0   11.46   
998      1614   Isabella Chen  Mechanical Keyboard       5.0   80.38   
999      1911     Samuel Hall           Smart Plug       2.0   17.73   

            order_date   region  
0           01/06/2023  Central  
1           09/05/2024    North  
2

In [155]:
print(df.columns)

Index(['order_id', 'customer_name', 'product', 'quantity', 'price',
       'order_date', 'region'],
      dtype='object')


In [156]:
print(df.isnull().sum())

order_id          0
customer_name    28
product           0
quantity         35
price             5
order_date        0
region            0
dtype: int64


# DATA ClEANING & TRANSFORMATION

1. Drop duplicates

In [157]:
df = df.drop_duplicates()

 2. Handle missing value

In [158]:
df['quantity'] = df['quantity'].fillna(1)
df = df.dropna(subset=['customer_name'])

3. Fix date formate

In [159]:
df['order_date'] = pd.to_datetime(
    df['order_date'],
    format='mixed'
)


4. Fix data types


In [160]:
df['price'] = pd.to_numeric(df['price'], errors='coerce')

**Check Missing Dates **

In [161]:
df['order_date'].isna().sum()

np.int64(0)

In [164]:
# Fix Missing Dates Properly
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')

# Data Engineering

In [165]:
# Create month from order date
df['month'] = df['order_date'].dt.strftime('%B')


In [166]:
# Create total revenue column
df['revenue'] = df['quantity'] * df['price']

In [167]:
# Categorize order size
def size_label(qty):
    if qty <= 2:
        return 'Small'
    if qty <= 5:
        return 'Medium'
    return 'Large'

df['order_size'] = df['quantity'].apply(size_label)

print(df[['revenue', 'month', 'order_size']].head())

   revenue      month order_size
0   127.86    January     Medium
1   248.30  September     Medium
2   761.84      March      Large
3   337.12   February      Large
4   506.45      April     Medium


# Load Into a Database

In [168]:
# Connect (creates file if not exists)
conn = sqlite3.connect('sales_dw.db')

In [169]:
# Write DataFrame to SQL table
df.to_sql(
name='orders',
con=conn,
if_exists='replace',  # or 'append'
index=False
)

925

In [170]:
# Verify the load
result = pd.read_sql(
'SELECT COUNT(*) as rows FROM orders',
conn
)
print(result)
conn.close()

   rows
0   925


In [171]:
import sqlite3, pandas as pd
conn = sqlite3.connect('sales_dw.db')
# Total revenue by region
q1 = '''
  SELECT region,
         SUM(revenue)   AS total_revenue,
         COUNT(*)       AS order_count,
         AVG(price)     AS avg_price
  FROM   orders
  GROUP  BY region
  ORDER  BY total_revenue DESC
'''
print(pd.read_sql(q1, conn))

      region  total_revenue  order_count  avg_price
0      North       48028.73          185  55.722235
1    Central       41450.55          203  47.368960
2       East       38801.70          169  53.358727
3       West       38458.86          201  46.971020
4      South       31210.36          155  50.580392
5  central          1243.82            4  51.210000
6     west           914.79            3  67.950000
7    north           694.42            3  30.620000
8     east           176.64            1  58.880000
9    south            35.08            1  35.080000
